### Import Libraries

In [31]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler

### Load Dataset

In [32]:
df = pd.read_csv("../data/traffic_dataset.csv")
print("Original Shape:", df.shape)
print("\nMissing Values:")
print(df.isnull().sum())

Original Shape: (120000, 15)

Missing Values:
timestamp                    0
hour                         0
day_of_week                  0
geohash_location             0
road_type                    0
num_lanes                    0
traffic_signals              0
large_vehicles_count         0
temperature                  0
humidity                     0
rainfall                     0
weather_conditions           0
nearby_landmarks         29994
event_indicator         113985
traffic_demand               0
dtype: int64


### Remove Duplicate Rows

In [33]:
duplicates = df.duplicated().sum()

print("Duplicate Rows:", duplicates)

df = df.drop_duplicates()

print("Shape After Removing Duplicates:", df.shape)

Duplicate Rows: 0
Shape After Removing Duplicates: (120000, 15)


### Validate Timestamp Format

In [34]:
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

print("Invalid timestamps:", df["timestamp"].isnull().sum())

Invalid timestamps: 0


### Extract Date Features

In [35]:
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["month"] = df["timestamp"].dt.month

df[["timestamp","hour","day_of_week","month"]].head()

,timestamp,hour,day_of_week,month
0,2023-01-01 00:00:00,0,6,1
1,2023-01-01 00:05:00,0,6,1
2,2023-01-01 00:10:00,0,6,1
3,2023-01-01 00:15:00,0,6,1
4,2023-01-01 00:20:00,0,6,1


### Separate Numerical And Categorical Columns

In [36]:
target = "traffic_demand"

numerical_cols = df.select_dtypes(include=np.number).columns.tolist()

if target in numerical_cols:
    numerical_cols.remove(target)

categorical_cols = df.select_dtypes(include="object").columns.tolist()

print("Numerical Columns")
print(numerical_cols)

print("\nCategorical Columns")
print(categorical_cols)

Numerical Columns
['hour', 'day_of_week', 'num_lanes', 'traffic_signals', 'large_vehicles_count', 'temperature', 'humidity', 'rainfall', 'month']

Categorical Columns
['geohash_location', 'road_type', 'weather_conditions', 'nearby_landmarks', 'event_indicator']


### Handle Missing Values

In [37]:
for col in numerical_cols:
    df[col].fillna(df[col].median(), inplace=True)

for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

print(df.isnull().sum())

timestamp                    0
hour                         0
day_of_week                  0
geohash_location             0
road_type                    0
num_lanes                    0
traffic_signals              0
large_vehicles_count         0
temperature                  0
humidity                     0
rainfall                     0
weather_conditions           0
nearby_landmarks         29994
event_indicator         113985
traffic_demand               0
month                        0
dtype: int64


### Outlier Treatment (Traffic Demand)

In [38]:
Q1 = df["traffic_demand"].quantile(0.25)
Q3 = df["traffic_demand"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print("Lower Bound:", lower)
print("Upper Bound:", upper)

df["traffic_demand"] = df["traffic_demand"].clip(lower, upper)

print(df["traffic_demand"].describe())

Lower Bound: -978.5
Upper Bound: 2441.5


count    120000.000000
mean        804.213933
std         625.453746
min         100.000000
25%         304.000000
50%         576.000000
75%        1159.000000
max        2441.500000
Name: traffic_demand, dtype: float64


### Label Encoding

In [39]:
le_road = LabelEncoder()
le_weather = LabelEncoder()

df["road_type"] = le_road.fit_transform(df["road_type"])

df["weather_conditions"] = le_weather.fit_transform(
    df["weather_conditions"]
)

le_geo = LabelEncoder()
df["geohash_location"] = le_geo.fit_transform(df["geohash_location"])

le_landmark = LabelEncoder()

df["nearby_landmarks"] = le_landmark.fit_transform(
    df["nearby_landmarks"]
)

In [40]:
import joblib

joblib.dump(le_geo, "../models/geohash_encoder.pkl")
joblib.dump(le_road, "../models/road_encoder.pkl")
joblib.dump(le_weather, "../models/weather_encoder.pkl")
joblib.dump(le_landmark, "../models/landmark_encoder.pkl")


['../models/landmark_encoder.pkl']

### One-Hot Encoding

In [41]:
df["event_indicator"] = df["event_indicator"].str.replace(" ", "_")

In [42]:
df = pd.get_dummies(
    df,
    columns=["event_indicator"],
    prefix="event",
    drop_first=False
)

print(df.shape)

(120000, 19)


### Feature Scaling

In [43]:

exclude_from_scaling = [
    "traffic_demand",
    "hour",
    "day_of_week",
    "num_lanes"
]

scale_columns = [
    col for col in numerical_cols
    if col not in exclude_from_scaling
]

scaler = StandardScaler()
df[scale_columns] = scaler.fit_transform(df[scale_columns])

### Final Dataset 

In [44]:
print("FINAL DATASET INFORMATION")
print("\nShape")
print(df.shape)

print("\nNull Values")
print(df.isnull().sum())

print("\nValue Ranges")

for col in scale_columns:
    print(f"\n{col}")
    print("Minimum:", round(df[col].min(),3))
    print("Maximum:", round(df[col].max(),3))

FINAL DATASET INFORMATION

Shape
(120000, 19)

Null Values
timestamp               0
hour                    0
day_of_week             0
geohash_location        0
road_type               0
num_lanes               0
traffic_signals         0
large_vehicles_count    0
temperature             0
humidity                0
rainfall                0
weather_conditions      0
nearby_landmarks        0
traffic_demand          0
month                   0
event_Concert           0
event_Conference        0
event_Festival          0
event_Sports_Event      0
dtype: int64

Value Ranges

traffic_signals
Minimum: -1.224
Maximum: 0.817

large_vehicles_count
Minimum: -2.83
Maximum: 5.658

temperature
Minimum: -2.843
Maximum: 2.719

humidity
Minimum: -2.002
Maximum: 2.561

rainfall
Minimum: -0.338
Maximum: 6.476

month
Minimum: -1.341
Maximum: 1.675


### Saving Preprocessed Dataset

In [45]:
os.makedirs("../data", exist_ok=True)

df.to_csv(
    "../data/traffic_preprocessed.csv",
    index=False
)
